# 🧬 Fine-Tuning Gemma 3 (4B) on CORD-19 using QLoRA

This notebook provides a complete pipeline to download a subset of the **CORD-19** (COVID-19 Open Research Dataset), preprocess it into instruction-following Q&A pairs, fine-tune the **Gemma 3 (4B)** model using QLoRA, and merge the weights.

### ⚡ Hardware Requirement
Ensure your Google Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU` or higher).

### 📦 1. Install Required Packages

In [ ]:
# Install the Hugging Face and PyTorch training stack
!pip install -q torch transformers datasets trl peft bitsandbytes accelerate huggingface_hub

### 🔑 2. Login to Hugging Face

Gemma 3 is a gated model. 
1. Go to [Hugging Face google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) and accept the license terms.
2. Go to your Hugging Face Settings -> Tokens and copy a **Read** token.
3. Run the cell below, paste your token, and log in.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### 📊 3. Download and Preprocess CORD-19 Data

We load a small slice of CORD-19 using streaming, format the abstract text, and save it locally as `cord19_instructions.json`.

In [ ]:
import json
from datasets import load_dataset

print("Downloading CORD-19 data...")
try:
    # Stream dataset to avoid downloading the full 100GB+ dataset
    dataset = load_dataset("allenai/cord19", "metadata", split="train", streaming=True)
except Exception as e:
    print("Error loading dataset, using medical QA fallback...")
    dataset = None

formatted_data = []
count = 0
limit = 2000  # Number of training samples

if dataset:
    for row in dataset:
        title = row.get("title", "").strip()
        abstract = row.get("abstract", "").strip()
        
        if title and abstract and len(abstract) > 100:
            entry = {
                "instruction": "Analyze the following scientific medical literature abstract and summarize the key findings, including any methodologies mentioned.",
                "input": f"Title: {title}\nAbstract: {abstract}",
                "output": f"This scientific paper investigated '{title}'. Key findings and summary:\n{abstract[:500]}..."
            }
            formatted_data.append(entry)
            count += 1
            if count >= limit:
                break
else:
    formatted_data = [
        {
            "instruction": "Analyze the following scientific medical literature abstract and summarize the key findings, including any methodologies mentioned.",
            "input": "Title: Clinical features of patients infected with 2019 novel coronavirus in Wuhan, China\nAbstract: A recent cluster of pneumonia cases in Wuhan, China, was caused by a novel coronavirus (2019-nCoV). We reports clinical features of 41 laboratory-confirmed patients. Common symptoms at onset of illness were fever, cough, and myalgia or fatigue.",
            "output": "This study details the clinical features of 41 laboratory-confirmed 2019-nCoV patients in Wuhan. The cohort was predominantly male, with less than half showing underlying health conditions. Key clinical symptoms at onset included fever, cough, and myalgia/fatigue."
        }
    ]

with open("cord19_instructions.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, indent=2, ensure_ascii=False)

print(f"Preprocessed dataset saved! Total items: {len(formatted_data)}")

### 🏋️ 4. Fine-Tune Gemma 3 (4B) via QLoRA

We load the model in 4-bit, configure LoRA adapters, map our dataset to the Gemma 3 chat template, and start training.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "google/gemma-3-4b-it"
DATASET_PATH = "cord19_instructions.json"
OUTPUT_DIR = "./gemma3-cord19-adapter"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 4-bit Quantization (saves GPU VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Format Prompts to Gemma 3 Chat Template
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_prompts(batch):
    formatted = []
    for inst, inp, out in zip(batch['instruction'], batch['input'], batch['output']):
        messages = [
            {"role": "system", "content": "You are a medical research AI assistant specializing in scientific literature."},
            {"role": "user", "content": f"{inst}\n\nContext:\n{inp}"},
            {"role": "assistant", "content": out}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append(text)
    return {"text": formatted}

dataset = dataset.map(format_prompts, batched=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,  # Run 100 training steps for demo
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=50,
    gradient_checkpointing=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

print("Training starting...")
trainer.train()

print(f"Saving adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training Complete!")

### 🔗 5. Merge Adapter Weights

Now we merge the LoRA weights back into the original 16-bit unquantized Gemma 3 base model.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "google/gemma-3-4b-it"
ADAPTER_DIR = "./gemma3-cord19-adapter"
MERGED_DIR = "./gemma3-cord19-merged"

print("Loading base model in FP16...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu"
)

print("Merging weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = model.merge_and_unload()

print(f"Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print("Model merged successfully!")

### 📦 6. Compress and Download Merged Model

We zip the merged model folder so you can download it to your local machine.

In [ ]:
!zip -r gemma3-cord19-merged.zip ./gemma3-cord19-merged
print("Finished zipping! Download 'gemma3-cord19-merged.zip' from the files sidebar on the left.")